## Loading the libraries

In [ ]:
!pip -q install transformers accelerate qwen-vl-utils pillow pandas tqdm scikit-learn

In [ ]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Enable gradient checkpointing to reduce memory usage (trades compute for memory)
model.gradient_checkpointing_enable()

processor = AutoProcessor.from_pretrained(MODEL_ID)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

...

chat_template.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Mount Google Drive and set paths

In [ ]:
# Mount Google Drive (for Colab)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Colab paths (Google Drive mounted at /content/drive)
SOFT_DIR = "/content/drive/MyDrive/CEE247C/247C Project/data/soft_story_images/images"
NON_DIR  = "/content/drive/MyDrive/CEE247C/247C Project/data/non_soft_story_images/images"

SOFT_DIAG = "/content/drive/MyDrive/CEE247C/247C Project/data/soft_story_images/diagrams"
NON_DIAG =  "/content/drive/MyDrive/CEE247C/247C Project/data/non_soft_story_images/diagrams"

In [ ]:
!nvidia-smi

Mon Mar  9 23:36:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   37C    P0             87W /  600W |   29181MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
def list_images(folder):
    exts = (".jpg", ".jpeg", ".png", ".webp")
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(exts)
    ])

# Only images directly inside each folder
soft_paths = list_images(SOFT_DIR)
non_paths = list_images(NON_DIR)
soft_diag_paths = list_images(SOFT_DIAG)
non_diag_paths = list_images(NON_DIAG)

print("Soft images:", len(soft_paths))
print("Non images:", len(non_paths))
print("Soft diagrams:", len(soft_diag_paths))
print("Non diagrams:", len(non_diag_paths))
print("Example soft path:", soft_paths[0] if soft_paths else "None found")
print("Example non path:", non_paths[0] if non_paths else "None found")

Soft images: 2214
Non images: 7348
Soft diagrams: 996
Non diagrams: 3259
Example soft path: /content/drive/MyDrive/CEE247C/247C Project/data/soft_story_images/images/0_view1.jpg
Example non path: /content/drive/MyDrive/CEE247C/247C Project/data/non_soft_story_images/images/1000_view0.jpg


In [ ]:
num_soft_buildings = len(soft_diag_paths)
num_non_buildings = len(non_diag_paths)
total_num_buildings = num_soft_buildings + num_non_buildings
print(f"Total buildings: {total_num_buildings} (Soft: {num_soft_buildings}, Non-soft: {num_non_buildings})")

Total buildings: 4255 (Soft: 996, Non-soft: 3259)


## Building-wise multi-view classification
Feed Qwen with multiple views for each building and the diagram for that building to classify soft story status.

In [ ]:
import re
from collections import defaultdict

def extract_building_number(filename):
    """Extract building number from filename like '12345_view_0.jpg' or '12345.jpg'"""
    basename = os.path.basename(filename)
    # Try to match building_number_view_X pattern first
    match = re.match(r'(\d+)_view_\d+', basename)
    if match:
        return match.group(1)
    # Try to match just building_number pattern (for diagrams)
    match = re.match(r'(\d+)', basename)
    if match:
        return match.group(1)
    return None

def group_images_by_building(image_paths, diagram_paths):
    """
    Group view images and diagrams by building number.
    Returns: dict with building_number as key and {'views': [...], 'diagram': '...'} as value
    """
    buildings = defaultdict(lambda: {'views': [], 'diagram': None})

    # Group view images
    for img_path in image_paths:
        building_num = extract_building_number(img_path)
        if building_num:
            buildings[building_num]['views'].append(img_path)

    # Assign diagrams
    for diag_path in diagram_paths:
        building_num = extract_building_number(diag_path)
        if building_num and building_num in buildings:
            buildings[building_num]['diagram'] = diag_path

    return dict(buildings)

# Group soft and non-soft buildings
soft_buildings = group_images_by_building(soft_paths, soft_diag_paths)
non_buildings = group_images_by_building(non_paths, non_diag_paths)

print(f"Soft story buildings with views: {len(soft_buildings)}")
print(f"Non-soft story buildings with views: {len(non_buildings)}")

# Check a sample building
sample_building = list(soft_buildings.keys())[0] if soft_buildings else None
if sample_building:
    print(f"\nSample building {sample_building}:")
    print(f"  Views: {len(soft_buildings[sample_building]['views'])}")
    print(f"  Diagram: {soft_buildings[sample_building]['diagram']}")

Soft story buildings with views: 950
Non-soft story buildings with views: 3224

Sample building 0:
  Views: 4
  Diagram: /content/drive/MyDrive/CEE247C/247C Project/data/soft_story_images/diagrams/0.png


In [ ]:
import random
# sample 500 buildings from each class for testing, taking all the views for those buildings, and the diagram
def sample_buildings(buildings_dict, num_samples=500):
    building_numbers = list(buildings_dict.keys())
    if len(building_numbers) < num_samples:
        print(f"Warning: Only {len(building_numbers)} buildings available, less than requested {num_samples}.")
        num_samples = len(building_numbers)
    # set a seed for reproducibility
    random.seed(19)
    sampled_numbers = random.sample(building_numbers, num_samples)
    return {num: buildings_dict[num] for num in sampled_numbers}

In [ ]:
sampled_soft_buildings = sample_buildings(soft_buildings, num_samples=500)
sampled_non_buildings = sample_buildings(non_buildings, num_samples=500)
print(f"Sampled {len(sampled_soft_buildings)} soft story buildings and {len(sampled_non_buildings)} non-soft story buildings for testing.")

Sampled 500 soft story buildings and 500 non-soft story buildings for testing.


# Zero Shot Learning

## Prompt for the multi-view images + diagram input for each building

In [ ]:
PROMPT = """
You are a structural classification assistant analyzing multiple Google Street View images of a single building and a targeting diagram.

YOUR TASK:
Classify whether the building exhibits a SOFT STORY condition using visible architectural evidence across all provided views.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ENGINEERING BASIS (ASCE 7)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
A soft story has lateral stiffness less than 70% of the story above.
Use solid wall area as a visual proxy for stiffness.
The test is RELATIVE: is floor 1 dramatically less enclosed than floor 2?

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WHAT REDUCES STIFFNESS (counts as opening)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✓ Vehicle-width garage openings / tuck-under parking
✓ Carport openings with no enclosing walls
✓ Floor-to-ceiling commercial glazing across most of the facade
✓ Pilotis with no solid infill walls
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WHAT DO NOT REDUCE STIFFNESS (not counted as opening)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✗ Entry doors, residential windows (any size)
✗ A single garage door with solid wall dominating both sides
✗ Decorative arches, recessed entries, stoops

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HOW TO CLASSIFY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Use the diagram to identify which facade each view captures.
2. Compare floor 1 vs. floor 2 enclosure across ALL views holistically.
3. A soft story requires a CLEAR, OBVIOUS discontinuity visible across
   multiple facades — if you have to squint a lot or guess, NO_CLEAR_CLASSIFICATION is better.
4. One open facade does not decide it — weigh the full visible perimeter.

SOFT_STORY:
  Floor 1 is dramatically less enclosed than floor 2 across most of the
  visible perimeter. The discontinuity is obvious and consistent across views.

NON_SOFT_STORY:
  Floor 1 is comparably enclosed to floor 2. Solid walls dominate the
  ground floor perimeter overall, even if one facade has some openings.

NO_CLEAR_CLASSIFICATION:
  Floor 1 vs. floor 2 cannot be reliably compared — ground floor hidden,
  too blurry, too far, or target building ambiguous across all views.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- If in doubt, classify as NO_CLEAR_CLASSIFICATION, not NON_SOFT_STORY.
- Do NOT classify on style, age, number of units, or retrofit status.
- Always compare floor 1 to floor 2 — never assess floor 1 in isolation.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Return exactly one of the following strings and nothing else:
SOFT_STORY
NON_SOFT_STORY
NO_CLEAR_CLASSIFICATION
"""

## Run classification on all buildings

## Condensed PROMPT for Few-Shot (reduces token count)

Currently same as PROMPT

In [ ]:
# PROMPT_CONDENSED = """
# You are a structural classification assistant analyzing Google Street View images of a building.

# YOUR TASK: Classify as SOFT_STORY or NON_SOFT_STORY. You must always return one of these two.

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# THE TEST (ASCE 7)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Compare floor 1 to floor 2. A soft story exists when floor 1 has less than
# 70% of the lateral stiffness of floor 2. Use solid wall area as a proxy.

# Counts as stiffness-reducing: vehicle-width garage bays, tuck-under parking,
# floor-to-ceiling commercial glazing, pilotis with no infill walls.

# Does NOT count: entry doors, residential windows, a single garage door in
# an otherwise solid facade, decorative arches, stoops.

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LABELS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SOFT_STORY: Floor 1 has dramatically less solid wall than floor 2.
#   Large vehicle-scale openings dominate the ground floor facade.
#   The discontinuity between floors is immediately obvious.

# NON_SOFT_STORY: Floor 1 and floor 2 are similarly enclosed.
#   Ground floor has normal doors, windows, at most one garage door.
#   THIS IS THE TYPICAL CASE FOR MOST BUILDINGS — when in doubt, choose this.

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FEW-SHOT EXAMPLES ABOVE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# The examples shown above are GROUND TRUTH — verified by structural engineers.
# Study the NON_SOFT_STORY examples carefully. Many normal-looking residential
# buildings with solid ground floors are NON_SOFT_STORY even if they have a
# garage door or large windows. Match the new building to the closest example.

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# OUTPUT FORMAT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Return exactly one of the following strings and nothing else:
# SOFT_STORY
# NON_SOFT_STORY
# """

In [ ]:
PROMPT_CONDENSED = """
You are a structural classification assistant analyzing Google Street View images of a building.

YOUR TASK: Classify the building as SOFT_STORY or NON_SOFT_STORY.
You must always return exactly one of these two labels.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
THE STRUCTURAL TEST (ASCE 7 CONCEPT)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Compare the ground floor (floor 1) to the floor above (floor 2).

A soft story occurs when the ground floor has substantially less lateral
stiffness than the floor above. Since stiffness cannot be measured directly
from an image, use visible solid wall area as a proxy.

If the ground floor is much more open than the floor above, the building
may be a soft-story structure.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FEATURES THAT REDUCE GROUND-FLOOR STIFFNESS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Examples include:
- multiple vehicle-width garage bays
- tuck-under parking
- open columns or pilotis with little or no infill walls
- large commercial storefront glazing
- ground floor dominated by large open bays

These features often create a clear vertical discontinuity between floors.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FEATURES THAT DO NOT INDICATE SOFT STORY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Do NOT classify a building as SOFT_STORY simply because it has:
- residential windows
- standard entry doors
- balconies or stoops
- decorative arches
- a single garage door in an otherwise solid facade

Many typical residential buildings include these elements and are still
NON_SOFT_STORY.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LABEL DEFINITIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SOFT_STORY:
The ground floor is clearly much more open than the floor above.
Large openings dominate the ground floor facade and the difference
between floors is visually obvious.

NON_SOFT_STORY:
The ground floor and the floor above are similarly enclosed.
The facade still contains substantial solid wall area even if
doors, windows, or one garage opening are present.

Most typical residential buildings fall into this category.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FEW-SHOT EXAMPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The examples shown above are ground truth verified by structural engineers.

Study both the SOFT_STORY and NON_SOFT_STORY examples carefully.
Focus on how the ground-floor enclosure compares to the floor above.

When analyzing a new building, compare it to the closest example.
If it visually resembles the SOFT_STORY examples, label SOFT_STORY.
If it resembles the NON_SOFT_STORY examples, label NON_SOFT_STORY.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Return exactly one of the following strings and nothing else:

SOFT_STORY
NON_SOFT_STORY
"""

In [ ]:
# Choose which prompt to use for few-shot
USE_CONDENSED_PROMPT = True  # Set to False to use full PROMPT
FEW_SHOT_PROMPT = PROMPT_CONDENSED if USE_CONDENSED_PROMPT else PROMPT

# Few Shot Version

In [ ]:
# ── Few-shot example folders ─────────────────────────────────────────────
# Each folder must follow the same naming convention as the main dataset:
#   images/  → {building_id}_view{N}.jpg  (or _view_{N}.jpg)
#   diagrams/ → {building_id}.png
#
# Put EXACTLY the buildings you want used as examples here.
# The code will pick the first FEW_SHOT_K from each class.

FS_SOFT_IMG_DIR  = "/content/drive/MyDrive/CEE247C/247C Project/data/few_shot/soft_story/images"
FS_SOFT_DIAG_DIR = "/content/drive/MyDrive/CEE247C/247C Project/data/few_shot/soft_story/diagrams"
FS_NON_IMG_DIR   = "/content/drive/MyDrive/CEE247C/247C Project/data/few_shot/non_soft_story/images"
FS_NON_DIAG_DIR  = "/content/drive/MyDrive/CEE247C/247C Project/data/few_shot/non_soft_story/diagrams"

FEW_SHOT_K = 3  # examples per class (1 soft + 1 non-soft = 2 total) - MINIMUM
MAX_VIEWS_PER_EXAMPLE =10  # Use only 1 view per example to minimize memory
MAX_QUERY_VIEWS = 10  # limit query building views to prevent OOM
SKIP_EXAMPLE_DIAGRAMS = False # Skip diagrams in examples to save memory (keep only query diagram)


## Load & Validate Few-Shot Examples

In [ ]:
def load_few_shot_examples(img_dir, diag_dir, label, k, max_views=None):
    """
    Load up to k buildings from img_dir/diag_dir.
    Returns a list of dicts: [{'views': [...], 'diagram': path, 'label': label}, ...]

    Args:
        max_views: If set, limit each example to this many views to reduce memory usage
    """
    buildings = group_images_by_building(
        list_images(img_dir),
        list_images(diag_dir)
    )
    examples = []
    for bid, data in list(buildings.items())[:k]:
        if not data['views']:
            print(f"  [SKIP] Building {bid} — no view images found")
            continue
        views = data['views'][:max_views] if max_views else data['views']
        examples.append({
            'building_id': bid,
            'views':       views,
            'diagram':     data['diagram'],
            'label':       label
        })
    return examples


fs_soft_examples = load_few_shot_examples(FS_SOFT_IMG_DIR,  FS_SOFT_DIAG_DIR,  'SOFT_STORY',     FEW_SHOT_K, MAX_VIEWS_PER_EXAMPLE)
fs_non_examples  = load_few_shot_examples(FS_NON_IMG_DIR,   FS_NON_DIAG_DIR,   'NON_SOFT_STORY', FEW_SHOT_K, MAX_VIEWS_PER_EXAMPLE)


# fs_soft_examples = load_few_shot_examples(FS_SOFT_IMG_DIR,  FS_SOFT_DIAG_DIR,  'SOFT_STORY',     FEW_SHOT_K)
# fs_non_examples  = load_few_shot_examples(FS_NON_IMG_DIR,   FS_NON_DIAG_DIR,   'NON_SOFT_STORY', FEW_SHOT_K)
few_shot_examples = fs_soft_examples + fs_non_examples

print(f"Few-shot examples loaded:")
print(f"  SOFT_STORY    : {len(fs_soft_examples)} buildings")
print(f"  NON_SOFT_STORY: {len(fs_non_examples)} buildings")

for ex in few_shot_examples:

    diag_ok = '✓' if ex['diagram'] else '✗ (missing)'
    print(f"  [{ex['label']}] id={ex['building_id']}  views={len(ex['views'])}  diagram={diag_ok}")

Few-shot examples loaded:
  SOFT_STORY    : 3 buildings
  NON_SOFT_STORY: 3 buildings
  [SOFT_STORY] id=461  views=2  diagram=✓
  [SOFT_STORY] id=579  views=2  diagram=✓
  [SOFT_STORY] id=617  views=2  diagram=✓
  [NON_SOFT_STORY] id=168  views=3  diagram=✓
  [NON_SOFT_STORY] id=27  views=2  diagram=✓
  [NON_SOFT_STORY] id=443  views=2  diagram=✓


## Few-Shot Message Builder & Classifier

In [ ]:
# ── Few-shot prompt header ────────────────────────────────────────────────
# Replaces the preamble used in zero-shot; the full PROMPT definition block
# is still appended at the end so all definitions/rules remain in context.

FEW_SHOT_INTRO = """\
You are a structural classification assistant.
Below are {n_soft} confirmed SOFT_STORY examples and {n_non} confirmed NON_SOFT_STORY examples.
Study each example carefully — pay attention to how open or enclosed the ground floor is
relative to the upper floors, and how the targeting diagram relates to each set of views.

After the examples you will be given a NEW building to classify.
Apply exactly the same reasoning to the new building.
"""

FEW_SHOT_EXAMPLE_HEADER = """\
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EXAMPLE {idx} — Label: {label}
  {n_views} street-view image(s) + 1 targeting diagram follow.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

FEW_SHOT_QUERY_HEADER = """\
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NEW BUILDING TO CLASSIFY
  {n_views} street-view image(s){diag_note} follow.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""


def create_few_shot_message(view_paths, diagram_path, examples, prompt):
    """
    Build a single Qwen2VL message that contains:
      1. Intro text explaining the few-shot setup
      2. For each example: header text + view images + diagram image
      3. Query header text + query view images + query diagram image
      4. The full PROMPT (definitions, rules, output format)

    All images are interleaved with text in one flat content list,
    which is how Qwen2VL expects multi-image multi-turn input.

    Args:
        view_paths   : list of str — paths to the query building's view images
        diagram_path : str or None — path to the query building's diagram
        examples     : list of dicts from load_few_shot_examples()
        prompt       : str — the full PROMPT string with definitions, rules, output format
    """
    content = []
    n_soft = sum(1 for e in examples if e['label'] == 'SOFT_STORY')
    n_non  = sum(1 for e in examples if e['label'] == 'NON_SOFT_STORY')

    # ── 1. Intro ─────────────────────────────────────────────────────────
    content.append({
        "type": "text",
        "text": FEW_SHOT_INTRO.format(n_soft=n_soft, n_non=n_non)
    })

    # ── 2. Examples (interleaved text + images) ───────────────────────────
    skip_example_diagrams = globals().get('SKIP_EXAMPLE_DIAGRAMS', False)

    for idx, ex in enumerate(examples, start=1):
        has_diag = ex['diagram'] and os.path.exists(ex['diagram']) and not skip_example_diagrams
        n_views  = len(ex['views'])

        # Header text for this example
        content.append({
            "type": "text",
            "text": FEW_SHOT_EXAMPLE_HEADER.format(
                idx=idx, label=ex['label'], n_views=n_views
            )
        })

        # View images
        for vp in ex['views']:
            content.append({"type": "image", "image": vp})

        # Diagram (skip if SKIP_EXAMPLE_DIAGRAMS is True to save memory)
        if has_diag:
            content.append({"type": "image", "image": ex['diagram']})

        # Confirmed label reminder (reinforces pattern)
        content.append({
            "type": "text",
            "text": f"Confirmed label for example {idx}: {ex['label']}\n"
        })

    # ── 3. Query building ─────────────────────────────────────────────────
    has_query_diag = diagram_path and os.path.exists(diagram_path)
    diag_note = " + 1 targeting diagram" if has_query_diag else " (no diagram available)"

    content.append({
        "type": "text",
        "text": FEW_SHOT_QUERY_HEADER.format(
            n_views=len(view_paths), diag_note=diag_note
        )
    })

    for vp in view_paths:
        content.append({"type": "image", "image": vp})

    if has_query_diag:
        content.append({"type": "image", "image": diagram_path})

    # ── 4. Full prompt (definitions + rules + output format) ──────────────
    # This ensures all classification rules from PROMPT are still applied
    content.append({"type": "text", "text": prompt})

    return [{"role": "user", "content": content}]


def classify_building_few_shot(view_paths, diagram_path, examples, prompt, model, processor):
    """
    Few-shot version of classify_building.
    Identical inference pipeline — only the message construction differs.

    Args:
        view_paths   : list of str — paths to the query building's view images
        diagram_path : str or None — path to the query building's diagram
        examples     : list of dicts from load_few_shot_examples()
        prompt       : str — the full PROMPT string with definitions, rules, output format
        model        : loaded Qwen2VL model
        processor    : loaded Qwen2VL processor

    Returns one of:
        'SOFT_STORY' | 'NON_SOFT_STORY' | 'NO_CLEAR_CLASSIFICATION'
        'PARSE_ERROR' | 'ERROR'
    """
    try:
        messages = create_few_shot_message(view_paths, diagram_path, examples, prompt)

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )
        inputs = inputs.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=16,
                do_sample=False,
                temperature=None,
                top_p=None,
            )
            generated_ids_trimmed = [
                out_ids[len(in_ids):]
                for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            output_text = processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )

        raw_response = output_text[0].strip()
        label = parse_response(raw_response)  # reuse zero-shot parser

        if label == "PARSE_ERROR":
            print(f"  [WARN] Unexpected model output: {repr(raw_response)}")

        return label

    except Exception as e:
        print(f"  [ERROR] {e}")
        return "ERROR"


def evaluate_buildings_few_shot(building_dict, ground_truth_label, examples, prompt, model, processor):
    """
    Few-shot equivalent of evaluate_buildings.
    Automatically excludes any building whose ID appears in `examples`
    to prevent evaluating on a building that was used as a demonstration.

    Args:
        building_dict:       {building_id: {'views': [...], 'diagram': path}}
        ground_truth_label:  expected label for all buildings in this dict
        examples:            list of few-shot example dicts (from load_few_shot_examples)
        prompt:              str — the full PROMPT string with definitions, rules, output format
        model, processor:    loaded Qwen2VL model and processor
    """
    # IDs used as few-shot examples — must not be evaluated
    example_ids = {str(ex['building_id']) for ex in examples}

    results = {}
    skipped = 0
    for building_id, data in building_dict.items():
        if str(building_id) in example_ids:
            skipped += 1
            continue

        # Limit views to prevent OOM (use MAX_QUERY_VIEWS if defined)
        query_views = data['views'][:MAX_QUERY_VIEWS] if 'MAX_QUERY_VIEWS' in globals() else data['views']

        print(f"  Classifying building {building_id}...")
        label = classify_building_few_shot(
            query_views, data['diagram'], examples, prompt, model, processor
        )
        results[building_id] = {
            'prediction':   label,
            'ground_truth': ground_truth_label,
            'correct':      label == ground_truth_label
        }
        print(f"    Prediction : {label}")
        print(f"    Ground truth: {ground_truth_label}")
        print(f"    Match      : {results[building_id]['correct']}")

        # Clear GPU cache periodically to prevent OOM
        if len(results) % 10 == 0:
            torch.cuda.empty_cache()

    correct = sum(r['correct'] for r in results.values())
    total   = len(results)
    print(f"\nSkipped {skipped} few-shot example building(s).")
    print(f"Accuracy: {correct}/{total} ({100 * correct / total:.1f}%)" if total else "No buildings evaluated.")
    return results


## Few-Shot Smoke Test

In [ ]:
# Clear GPU cache before few-shot inference to free up memory
import gc
torch.cuda.empty_cache()
gc.collect()

# ── Smoke test: one soft-story building (not in examples) ─────────────────
# Find the first non_soft building whose ID is NOT one of the few-shot examples
example_ids = {str(ex['building_id']) for ex in few_shot_examples}
test_id = next(
    bid for bid in non_buildings
    if str(bid) not in example_ids
)
test_data = non_buildings[test_id]

# Limit query views to prevent OOM
query_views = test_data['views'][:MAX_QUERY_VIEWS]

print(f"Few-shot smoke test on building {test_id}...")
print(f"  Using {len(query_views)} views, {len(few_shot_examples)} examples")
result = classify_building_few_shot(
    query_views, test_data['diagram'], few_shot_examples, FEW_SHOT_PROMPT, model, processor

)

print(f"Prediction   : {result}")
print(f"Ground truth : NON_SOFT_STORY")

Few-shot smoke test on building 1000...
  Using 2 views, 6 examples
Prediction   : SOFT_STORY
Ground truth : NON_SOFT_STORY


## Few-Shot Full Evaluation

In [ ]:
# Clear GPU cache before starting evaluation
torch.cuda.empty_cache()
gc.collect()

print("\n" + "=" * 60)
print("FEW-SHOT: EVALUATING NON-SOFT STORY BUILDINGS")
print("=" * 60)
fs_non_results = evaluate_buildings_few_shot(
    sampled_non_buildings, "NON_SOFT_STORY", few_shot_examples, FEW_SHOT_PROMPT, model, processor
)
print("=" * 60)
print("FEW-SHOT: EVALUATING SOFT STORY BUILDINGS")
print("="  * 60)
fs_soft_results = evaluate_buildings_few_shot(
    sampled_soft_buildings, "SOFT_STORY", few_shot_examples, FEW_SHOT_PROMPT, model, processor
)






# ── Combine into DataFrame ────────────────────────────────────────────────
fs_all_results = []
for building_id, data in {**fs_soft_results, **fs_non_results}.items():
    fs_all_results.append({
        'building_number': building_id,
        'predicted':       data['prediction'],
        'ground_truth':    data['ground_truth'],
        'correct':         data['correct']
    })

fs_results_df = pd.DataFrame(fs_all_results)
print(f"\n{'=' * 60}")
print(f"Total buildings evaluated : {len(fs_results_df)}")
print(f"Overall accuracy          : {fs_results_df['correct'].mean():.1%}")
print(f"\nPrediction distribution:")
print(fs_results_df['predicted'].value_counts().to_string())
print(f"\nAccuracy by category:")
for gt in fs_results_df['ground_truth'].unique():
    sub = fs_results_df[fs_results_df['ground_truth'] == gt]
    print(f"  {gt}: {sub['correct'].sum()}/{len(sub)} ({sub['correct'].mean():.1%})")



FEW-SHOT: EVALUATING NON-SOFT STORY BUILDINGS
  Classifying building 591...
    Prediction : SOFT_STORY
    Ground truth: NON_SOFT_STORY
    Match      : False
  Classifying building 1166...
    Prediction : SOFT_STORY
    Ground truth: NON_SOFT_STORY
    Match      : False
  Classifying building 992...
    Prediction : SOFT_STORY
    Ground truth: NON_SOFT_STORY
    Match      : False
  Classifying building 2943...
    Prediction : SOFT_STORY
    Ground truth: NON_SOFT_STORY
    Match      : False
  Classifying building 1454...
    Prediction : NON_SOFT_STORY
    Ground truth: NON_SOFT_STORY
    Match      : True
  Classifying building 290...
    Prediction : SOFT_STORY
    Ground truth: NON_SOFT_STORY
    Match      : False
  Classifying building 1751...
    Prediction : SOFT_STORY
    Ground truth: NON_SOFT_STORY
    Match      : False
  Classifying building 2468...
    Prediction : SOFT_STORY
    Ground truth: NON_SOFT_STORY
    Match      : False
  Classifying building 2298...
  

In [ ]:
fs_output_path = "/content/drive/MyDrive/CEE247C/247C Project/data/qwen_early_fusion_few_shot_results.csv"
fs_results_df.to_csv(fs_output_path, index=False)
print(f"Few-shot results saved to: {fs_output_path}")


Few-shot results saved to: /content/drive/MyDrive/CEE247C/247C Project/data/qwen_early_fusion_few_shot_results.csv


In [ ]:
fs_output_path = "/content/drive/MyDrive/CEE247C/247C Project/data/qwen_early_fusion_few_shot_results_updated_prompt_for_more_NON_SOFT.csv"
fs_results_df.to_csv(fs_output_path, index=False)
print(f"Few-shot results saved to: {fs_output_path}")
